In [1]:
%pwd

'c:\\Users\\Dhanush Ramachandran\\Desktop\\Dhanush --\\Personal DS\\MLops\\end-to-end project\\project\\research'

In [2]:
import os
os.chdir("../")

In [3]:
# design of data ingestion
from dataclasses import dataclass
from pathlib import Path
 

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
# config manager
from src.constants import *
from src.utils.common import *
from pathlib import Path 
#read_yaml(Path("config.yaml"))

class ConfigurationManager:
    def __init__(self,config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH ):
        print(config_file_path)
        self.configs = read_yaml(config_file_path)
        #self.params = read_yaml_file(params_file_path)
        #self.schema = read_yaml_file(schema_file_path)

        create_directories([self.configs.artifact_root])

    def get_data_ingestion_config(self)->DataIngestionConfig:
        config = self.configs.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_url = config.source_url,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config
    

# unit test
config = ConfigurationManager()
data_ingestion_config = config.get_data_ingestion_config()
print(data_ingestion_config)

config.yaml
reading yaml file:  config.yaml
file path:  config.yaml


TypeError: Logger.log() missing 1 required positional argument: 'msg'

In [19]:
# use data ingestion configs
#from box import ConfigBox
from box.exceptions  import BoxValueError
import pandas as pd
#from src import logger
class DataIngestionLoader:
    def __init__(self, config:DataIngestionConfig):
        self.config = config
        
    def load_data(self):
        # load from source url if source url is given
        if self.config.source_url:
            logger.log(f"Loading data from source url: {self.config.source_url}")
            try:
                df = pd.read_csv(self.config.source_url)
                logger.log(f"Data read successfully from {self.config.source_url}")
            except Exception as e:
                logger.log(f"Error reading CSV file from source url: {e}")
                raise BoxValueError(f"Error reading CSV file: {e}")
        else:
            # load from local data file
            try:
                logger.log(f"Loading data from local file: {self.config.local_data_file}")
                df = pd.read_csv(self.config.local_data_file)
            except Exception as e:
                logger.log(f"Error reading CSV file from local file: {e}")
                raise BoxValueError(f"Error reading csv file: {e}")
            
        return df

In [22]:
# ingestion executor
class DataIngestionExecutor:
    def __init__(self):
        conf_manager = ConfigurationManager()
        self.config = conf_manager.get_data_ingestion_config()
        self.data_loader = DataIngestionLoader(self.config)
    def initiate_data_ingestion(self)->pd.DataFrame:
        logger.log("Data ingestion execution started")
        df = self.data_loader.load_data()
        logger.log("Data ingestion process completed")
        logger.log(f"Data Shape: {df.shape}")
        logger.log(f"Data cols: {df.columns}")
        logger.log(f"Df Dtypes: {df.dtypes}")
        logger.log(f"Data Head: {df.head()}")
        return df
    
if __name__ == "__main__":
    ingestion_executor =  DataIngestionExecutor()
    df  = ingestion_executor.initiate_data_ingestion()
    

AttributeError: 'WindowsPath' object has no attribute 'read'